# Restoring Historical Photographs — Notebook 4: Integration and Deployment

**AAI-521 Group 3** · Nolan Robbins · Sean McGowan · Jackson Kenyon

The four fine-tuned models are integrated into one system (`restoration.pipeline`) and
served by a **Flask** web application (`restoration.webapp`). Users can:

1. upload a photograph (JPEG/PNG/TIFF, ≤ 25 MB; downscaled to 2048 px for processing);
2. paint over damaged areas with a brush, or let a heuristic scratch detector find them;
3. choose any subset of **denoise → inpaint → colorize → super-resolve** (always run in
   that order);
4. compare before/after with a slider, inspect each intermediate stage and download the
   full-resolution result;
5. toggle **fine-tuned vs pretrained** weights to see what fine-tuning changed;
6. rate the result. Ratings are stored for the user-testing analysis below.

| Layer | Responsibility |
|---|---|
| `webapp/templates/index.html` | UI: upload, mask canvas, task selection, comparison slider, feedback form |
| `webapp/__init__.py` (Flask) | HTTP only: validation, image encoding, a GPU lock, feedback logging |
| `pipeline.py` | Business logic: lazy model loading, fine-tuned-or-pretrained fallback, task order |
| `models/inference.py` | Per-task pre/post-processing shared with training and evaluation |

## Environment setup

Runs unchanged on **Google Colab** or locally.

* **Colab:** clones the repo, installs it, mounts Google Drive and copies the packed
  dataset (`MyDrive/aai521/aai521_dataset.tar`, made by `scripts/pack_dataset.py`) to the
  VM's fast local disk. Checkpoints are written to `MyDrive/aai521/checkpoints`, so they
  survive disconnects.
* **Locally:** uses `$AAI521_DATA_ROOT` from `.env`.

All project logic lives in the importable `restoration` package (`src/restoration`), so
the notebooks stay focused on the narrative, results and figures.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/NolanRobbins/AAI-521-Final-Project.git /content/AAI-521-Final-Project 2>/dev/null || git -C /content/AAI-521-Final-Project pull -q
    %cd /content/AAI-521-Final-Project
    !pip install -q -r requirements.txt && pip install -q -e .
    sys.path.insert(0, "/content/AAI-521-Final-Project/src")  # editable install, no restart needed

from restoration import colab

paths = colab.setup()  # must run before importing model code (sets HF_HOME on Colab)
paths

## Launch the app

On Colab, the app runs in a background thread and is exposed through Colab's
authenticated port proxy (no third-party tunnel). Locally, open the printed URL. The
first request loads the selected models (about 10–30 s); later requests are much faster.

In [ ]:
import threading
import time

from restoration.webapp import create_app

PORT = 5000
app = create_app()
server = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT, threaded=True, use_reloader=False), daemon=True)
server.start()
time.sleep(2)

if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/")
else:
    print(f"Open http://127.0.0.1:{PORT}")

## Programmatic use (same pipeline, no UI)

The same pipeline is available as a Python API. Here it restores a real damaged FSA/OWI
black-and-white negative from our evaluation pool: denoise, auto-detected scratch
inpainting, colorization and ×4 super-resolution.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

from restoration.config import raw_dir
from restoration.data import degradations as deg
from restoration.evaluation.plotting import apply_style, image_grid
from restoration.pipeline import RestorationPipeline, TASK_ORDER

apply_style()
pool = pd.read_csv(raw_dir("loc_fsa_bw") / "manifest.csv")
row = pool[pool["status"] == "ok"].sample(1, random_state=3).iloc[0]
photo = deg.resize_max_side(deg.to_float(np.asarray(Image.open(raw_dir("loc_fsa_bw") / row["path"]).convert("RGB"))), 512)

pipe = RestorationPipeline()
result = pipe.run(photo, TASK_ORDER, auto_mask=True)
print(row["title"][:100]); print("weights:", result.weights); print("seconds:", {k: round(v, 1) for k, v in result.seconds.items()})
image_grid([photo, result.mask.astype(float) if result.mask is not None else np.zeros(photo.shape[:2])]
           + [result.stages[t] for t in TASK_ORDER if t in result.stages],
           ["input (real damage)", "detected damage"] + [t for t in TASK_ORDER if t in result.stages], ncols=3, size=4.2);

## User testing

Testers (classmates and family members with old photos) used the app and rated each
result 1–5, overall and per task, with optional comments. Ratings are appended to
`$AAI521_DATA_ROOT/outputs/user_feedback.csv`.

In [ ]:
from restoration.config import data_root

fb_path = data_root() / "outputs" / "user_feedback.csv"
if fb_path.is_file():
    fb = pd.read_csv(fb_path)
    rating_cols = ["overall", "denoise", "inpaint", "colorize", "super_resolution"]
    print(f"{len(fb)} feedback submissions")
    display(fb[rating_cols].apply(pd.to_numeric, errors="coerce").agg(["count", "mean", "std"]).round(2))
    display(fb.loc[fb["comment"].fillna("").str.len() > 0, ["timestamp", "tasks", "overall", "comment"]].tail(10))
else:
    print("No feedback collected yet: run the app and submit ratings.")